In [9]:
from dotenv import load_dotenv
from pathlib import Path
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import (
    TextLoader, PyPDFLoader, BSHTMLLoader
)
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage
import os
import shutil

In [2]:
load_dotenv()
os.environ['ANONYMIZED_TELEMETRY'] = 'False'

Load the documents from the data folder

In [3]:
data_dir = Path(r'C:\Users\USER\rag_course\04_data_ingestion_document_processing\data') 

files = {
    'agriculture.html': 'crops',
    'agriculture.txt': 'crops',
    'crop_disease.pdf': 'crops',
    'nigeria_health_diseases_and_prevention.pdf': 'health',
}

docs = []

# Loop through each file
for name, topic in files.items():
    
    # Build full path to the file
    path = data_dir / name
    print(f'Loading {name} (topic={topic})...')
    
    # Pick the right loader based on file type
    if path.suffix == '.html':
        loader = BSHTMLLoader(str(path), open_encoding='utf-8', bs_kwargs={'features': 'html.parser'})
        
    elif path.suffix == '.txt':
        loader = TextLoader(str(path), encoding='utf-8')
        
    elif path.suffix == '.pdf':
        loader = PyPDFLoader(str(path))
        
    # Returns a list of Document objects
    loaded = loader.load()
    
    # Summary of this file
    print(f'    Loaded {len(loaded)} document(s) from {name}')
    
    # Add domain metadata(health, crops)
    for d in loaded:
        d.metadata['domain'] = topic
        d.metadata['file_name'] = name
        
    docs.extend(loaded)
       
print(f'\nTotal documents loaded: {len(docs)}')
print('Topics presents:', sorted({d.metadata['domain'] for d in docs}))

# Inspect one document
print('\nSample document metadata:')
print(docs[0].metadata)
print(docs[0].page_content[:300])

Loading agriculture.html (topic=crops)...
    Loaded 1 document(s) from agriculture.html
Loading agriculture.txt (topic=crops)...
    Loaded 1 document(s) from agriculture.txt
Loading crop_disease.pdf (topic=crops)...
    Loaded 29 document(s) from crop_disease.pdf
Loading nigeria_health_diseases_and_prevention.pdf (topic=health)...
    Loaded 6 document(s) from nigeria_health_diseases_and_prevention.pdf

Total documents loaded: 37
Topics presents: ['crops', 'health']

Sample document metadata:
{'source': 'C:\\Users\\USER\\rag_course\\04_data_ingestion_document_processing\\data\\agriculture.html', 'title': 'Agriculture in Nigeria: Crops, Livestock, and Disease Management', 'domain': 'crops', 'file_name': 'agriculture.html'}




Agriculture in Nigeria: Crops, Livestock, and Disease Management



Agriculture in Nigeria
Last updated: August 2026


Introduction

            Agriculture is one of the most important sectors of the Nigerian economy. It contributes significantly to the country

Split into chunks

In [4]:
# Create the splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=900,
    chunk_overlap=150,
    separators=['\n\n', '\n', '. ', '? ', '! ']  # split on natural sentence breaks
)

# Split all docs into chunks 
chunks = splitter.split_documents(docs)

print(f'Total chunks created: {len(chunks)}')
print(f'First chunk topic: {chunks[0].metadata.get("topic")}')
print(f'First chunk file: {chunks[0].metadata.get("file_name")}')

from collections import Counter
domain_counts = Counter(c.metadata['domain'] for c in chunks)
print('Chunk per doamin:', dict(domain_counts))

Total chunks created: 80
First chunk topic: None
First chunk file: agriculture.html
Chunk per doamin: {'crops': 33, 'health': 47}


Build the tagged vector store

In [5]:
# Absolute path for the new domain-tagged store
persist_dir = r'C:\Users\USER\rag_course\chroma_db_domain'

# Delete the folder if it exists
if Path(persist_dir).exists():
    shutil.rmtree(persist_dir)
    print(f'Removed old store at {persist_dir}')
    
# Build the store — embeddings 
embeddings = OpenAIEmbeddings()

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=persist_dir
)

print(f'Vector store built with {vectorstore._collection.count()} chunks')
print(f'   Location: {persist_dir}')

Removed old store at C:\Users\USER\rag_course\chroma_db_domain


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Vector store built with 80 chunks
   Location: C:\Users\USER\rag_course\chroma_db_domain


Inspect the embeddings

In [6]:
# Fetch everything including embeddings and metadata
data = vectorstore.get(include=['embeddings', 'documents', 'metadatas'])

print(f'Total vectors: {len(data["embeddings"])}')
print(f'Embedding dimension: {len(data["embeddings"][0])}')
print()

# Show the first 3 embeddings (first 8 numbers only)
for i in range(3):
    print(f'--- Vector {i} ---')
    print('Domain:', data['metadatas'][i].get('domain'))
    print('File:  ', data['metadatas'][i].get('file_name'))
    print('Text:  ', data['documents'][i][:80], '...')
    print('Vector (first 8 numbers):', data['embeddings'][i][:8])
    print()

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


Total vectors: 80
Embedding dimension: 1536

--- Vector 0 ---
Domain: health
File:   nigeria_health_diseases_and_prevention.pdf
Text:   government but its response system is slow. The role of 
the department of publi ...
Vector (first 8 numbers): [-0.004568132571876049, 0.010306557640433311, -0.030214836820960045, -0.019748695194721222, 0.0023023521061986685, -0.005286266561597586, -0.0009949152590706944, 0.012321323156356812]

--- Vector 1 ---
Domain: crops
File:   crop_disease.pdf
Text:   Deformation of organs
O A lot of pathogens often cause permanent 
damages to the ...
Vector (first 8 numbers): [-0.009125955402851105, -0.0025629710871726274, 0.015789350494742393, -0.01808071695268154, -0.034502167254686356, 0.0375046469271183, -0.011463411152362823, 4.609068128047511e-05]

--- Vector 2 ---
Domain: crops
File:   crop_disease.pdf
Text:   - Avoidance: In this case crops are grown in a 
manner that avoids exposing them ...
Vector (first 8 numbers): [-0.012125947512686253, -0.004302650

Plain retrieval — no filter (see the imbalance)


In [7]:
# Create a retriever — no filter
retriever = vectorstore.as_retriever(search_kwargs={'k': 8})

# Ask a health question
query = 'What are the top causes of death in Nigeria?'
docs = retriever.invoke(query)

print(f'Query: {query}\n')
print(f'Retrieved {len(docs)} docs:\n')

for i, d in enumerate(docs):
    domain = d.metadata.get('domain', '?')
    source = d.metadata.get('file_name', '?')
    print(f'--- Doc {i} [domain={domain}, file={source}] ---')
    print(d.page_content[:200])
    print()

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Query: What are the top causes of death in Nigeria?

Retrieved 8 docs:

--- Doc 0 [domain=health, file=nigeria_health_diseases_and_prevention.pdf] ---
infectious diseases, sewage disposal, health insurance, 
water supply, air pollution, noise pollution, environmen-
tal radiation, housing, solid waste disposal, disaster 
management, control of vector

--- Doc 1 [domain=health, file=nigeria_health_diseases_and_prevention.pdf] ---
 The top causes of death in Nigeria are; malaria, 
lower respiratory infections, HIV/AIDS,      
diarrheal diseases, road injuries, protein -energy 
malnutrition, cancer, meningitis, stroke and 
tube

--- Doc 2 [domain=health, file=nigeria_health_diseases_and_prevention.pdf] ---
health care services, brain drain, and irrational          
appointment of health workers among others. A new 
global burden has revealed that malaria and HIV are 
still leading cause of death in Nige

--- Doc 3 [domain=health, file=nigeria_health_diseases_and_prevention.pdf] ---
diseas

Test a crop question

In [8]:
# Crop question
query = 'What are the common crop diseases and how are they controlled?'
docs = retriever.invoke(query)

print(f'Query: {query}\n')
for i, d in enumerate(docs):
    domain = d.metadata.get('domain', '?')
    print(f'--- Doc {i} [domain={domain}] ---')
    print(d.page_content)
    print()

Query: What are the common crop diseases and how are they controlled?

--- Doc 0 [domain=crops] ---
Common Crop Diseases and Their Control

1. Cassava Mosaic Disease
Affected Crop: Cassava
Symptoms: Yellowing and mottling of leaves, stunted growth, reduced yield.
Control/Cure: Use disease-free cuttings, plant resistant varieties, and remove infected plants early.


2. Maize Smut
Affected Crop: Maize
Symptoms: Large grey or black galls on ears, stalks, and leaves.
Control/Cure: Remove and destroy infected plants, rotate crops, and plant resistant hybrids.


3. Rice Blast
Affected Crop: Rice
Symptoms: Spindle-shaped lesions on leaves, panicle damage, and reduced grain quality.
Control/Cure: Use resistant varieties, avoid excessive nitrogen fertiliser, and apply recommended fungicides.

--- Doc 1 [domain=crops] ---
Fisheries and aquaculture provide employment and protein for many families. Catfish and tilapia are commonly farmed in ponds and tanks.

Common Crop Diseases and Control

1. Ca

In [13]:
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

# Prompt 1: Rewrite follow-up questions into standalone form
rewrite_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'You rewrite a user\'s latest question into a standalone question using the chat history.\n'
     '\n'
     'Rules:\n'
     '1. If the latest question refers to something from the history (like "the first one", '
     '"that", "it", "the second option"), REPLACE that reference with the actual item from the history.\n'
     '2. Do NOT answer the question. Only rewrite it.\n'
     '3. If the question is already standalone, return it unchanged.\n'
     '\n'
     'Examples:\n'
     '- History mentions "Malaria" first, then "HIV/AIDS".\n'
     '  User: "Tell me more about the first one."\n'
     '  Rewritten: "Tell me more about malaria."\n'
     '\n'
     '- History mentions "Cassava Mosaic Disease" first, then "Maize Smut".\n'
     '  User: "How do I control the second one?"\n'
     '  Rewritten: "How do I control Maize Smut?"'),
    MessagesPlaceholder('chat_history'),
    ('human', '{input}'),
])

# Prompt 2: Answer using retrieved context
qa_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant. '
               'Answer the question using only the provided context. '
               'If you don\'t know, say you don\'t know.'),
    ('human', 'Context:\n{context}\n\nQuestion: {input}'),
])

print('Prompts ready')

Prompts ready


Build the reusable chains

In [14]:
# Reusable chains — build once, invoke many times
rewrite_chain = rewrite_prompt | llm | StrOutputParser()
qa_chain = qa_prompt | llm | StrOutputParser()

print('Chains ready')

Chains ready


Define the helper functions


In [15]:
def rewrite_question(question, history):
    '''Turn a follow-up into a standalone question using chat history.'''
    
    return rewrite_chain.invoke({
        'chat_history': history,
        'input': question,
    })

def generate_answer(question, context):
    '''Generate the final answer from retrieved context.'''
    
    return qa_chain.invoke({
        'context': context,
        'input': question,
    })
    
print('Helper ready')

Helper ready


Define the session store and ask() function

In [18]:
store = {}

# session_id → list of messages
def get_history(session_id):
    '''Return this session's message list, creating it if needed.'''
    
    # If session is new
    if session_id not in store:
        
        # Create empty history
        store[session_id] = []
    return store[session_id]

def ask(question, session_id):
    '''Answer a question with conversation history support.'''
    
    # 1. Fetch this session's history
    history = get_history(session_id)
    
    # 2. Rewrite the question using history (only if history exists)
    rewritten = rewrite_question(question, history) if history else question
    
    # 3. Retrieve relevant documents
    docs = retriever.invoke(rewritten)
    
    # 4. Join the chunk texts into one context block
    context = '\n\n'.join(d.page_content for d in docs)
    
    # 5. Generate the answer from the context
    answer = generate_answer(rewritten, context)
    
    # Step 6: save this turn into history
    history.append(HumanMessage(content=question))   # user's original question.
    history.append(AIMessage(content=answer))   # Save assistant's answer.
        
    return answer
    
print('Conversational helper ready')

Conversational helper ready


Test Turn 1 (health question)

In [19]:
session_id = 'test-1'

q1 = 'What are the top causes of death in Nigeria?'
a1 = ask(q1, session_id)

print('👤 User:', q1)
print('🤖 Assistant:', a1)
print('-' * 70)

👤 User: What are the top causes of death in Nigeria?
🤖 Assistant: The top causes of death in Nigeria are:

1. Malaria
2. Lower Respiratory Infections
3. HIV/AIDS
4. Diarrheal Diseases
5. Road Injuries
6. Protein-energy malnutrition
7. Cancer
8. Meningitis
9. Stroke
10. Tuberculosis
----------------------------------------------------------------------


Test Turn 2 (vague follow-up)

In [20]:
# Same session_id so the history is shared
q2 = 'Tell me more about the first one.'
a2 = ask(q2, session_id)

print('👤 User:', q2)
print('🤖 Assistant:', a2)
print('-' * 70)

👤 User: Tell me more about the first one.
🤖 Assistant: Malaria remains the foremost killer disease in Nigeria, with an estimated 300,000 children dying from it each year. It accounts for over 25% of infant mortality (children under one year), 30% of childhood mortality (children under five), and 11% of maternal mortality. At least 50% of the population experiences at least one episode of malaria annually, while children under five have 2 to 4 attacks each year. Malaria is particularly severe among pregnant women and young children due to their lower levels of immunity.

To combat malaria, the Society for Family Health (SFH) focuses on treatment and prevention through Pre-Packaged Therapy (PPT) and Long Lasting Insecticide Treated Nets (LLINs). The Federal Ministry of Health has implemented a new treatment policy that includes Artemisinin-based Combination Therapy (ACT) as the first-line drug for treating uncomplicated malaria. A specific brand of ACT for children, called KidACT, was de

Test Turn 3 (crop follow-up — switch topic)

In [21]:
# Fresh session for a different topic
session_id_crop = 'test-crop-1'
q3 = 'What are the common crop diseases and how are they controlled?'
a3 = ask(q3, session_id_crop)

print('👤 User:', q3)
print('🤖 Assistant:', a3)
print('-' * 70)

👤 User: What are the common crop diseases and how are they controlled?
🤖 Assistant: The common crop diseases and their control methods are:

1. **Cassava Mosaic Disease**
   - **Affected Crop:** Cassava
   - **Symptoms:** Yellowing and mottling of leaves, stunted growth, reduced yield.
   - **Control/Cure:** Use disease-free cuttings, plant resistant varieties, and remove infected plants early.

2. **Maize Smut**
   - **Affected Crop:** Maize
   - **Symptoms:** Large grey or black galls on ears, stalks, and leaves.
   - **Control/Cure:** Remove and destroy infected plants, rotate crops, and plant resistant hybrids.

3. **Rice Blast**
   - **Affected Crop:** Rice
   - **Symptoms:** Spindle-shaped lesions on leaves, panicle damage, and reduced grain quality.
   - **Control/Cure:** Use resistant varieties, avoid excessive nitrogen fertiliser, and apply recommended fungicides.

4. **Tomato Leaf Curl Virus**
   - **Affected Crop:** Tomatoes and peppers
   - **Symptoms:** Curling and yellowi

Inspect the saved history

In [23]:
history = store['test-1']

print(f'History for session "test-1" {len(history)} messages\n')

for i, msg in enumerate(history):
    role = '👤 User' if msg.type == 'human' else '🤖 Assistant'
    print(f'[{i}] {role}: {msg.content}')
    print()

History for session "test-1" 4 messages

[0] 👤 User: What are the top causes of death in Nigeria?

[1] 🤖 Assistant: The top causes of death in Nigeria are:

1. Malaria
2. Lower Respiratory Infections
3. HIV/AIDS
4. Diarrheal Diseases
5. Road Injuries
6. Protein-energy malnutrition
7. Cancer
8. Meningitis
9. Stroke
10. Tuberculosis

[2] 👤 User: Tell me more about the first one.

[3] 🤖 Assistant: Malaria remains the foremost killer disease in Nigeria, with an estimated 300,000 children dying from it each year. It accounts for over 25% of infant mortality (children under one year), 30% of childhood mortality (children under five), and 11% of maternal mortality. At least 50% of the population experiences at least one episode of malaria annually, while children under five have 2 to 4 attacks each year. Malaria is particularly severe among pregnant women and young children due to their lower levels of immunity.

To combat malaria, the Society for Family Health (SFH) focuses on treatment and 